In [1]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import mediapipe as mp
import random

# Define your class mappings
class_map = {'punch': 0, 'kick': 1, 'downtime': 2}

# Set up MediaPipe for pose detection
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

class PunchingBagDataset(Dataset):
    def __init__(self, data_dir, max_frames=16, frame_size=(64, 64), transform=None):
        self.data_dir = data_dir
        self.max_frames = max_frames  # Max number of frames to extract per video
        self.frame_size = frame_size  # Size to resize each frame
        self.transform = transform
        self.data = []
        
        # Collect all video files and labels based on folders
        for class_name, class_idx in class_map.items():
            class_folder = os.path.join(data_dir, class_name)
            for file_name in os.listdir(class_folder):
                if file_name.endswith('.mov'):
                    self.data.append((os.path.join(class_folder, file_name), class_idx))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        video_path, label = self.data[idx]
        frames = []
        pose_keypoints = []

        # Load the video using OpenCV
        cap = cv2.VideoCapture(video_path)
        frame_count = 0

        while cap.isOpened() and frame_count < self.max_frames:
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to RGB and then to float32 with normalization
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_resized = cv2.resize(frame_rgb, self.frame_size)
            frame_tensor = torch.tensor(frame_resized).permute(2, 0, 1).float() / 255.0  # Convert to float32 and normalize
            frames.append(frame_tensor)

            # Perform pose estimation
            results = pose.process(frame_rgb)
            
            if results.pose_landmarks:
                # Extract keypoints (x, y coordinates)
                keypoints = torch.tensor([(lm.x, lm.y) for lm in results.pose_landmarks.landmark], dtype=torch.float32)
            else:
                keypoints = torch.zeros((33, 2), dtype=torch.float32)  # Placeholder if no pose detected
            pose_keypoints.append(keypoints)

            frame_count += 1

        cap.release()

        # Pad frames and keypoints if less than max_frames
        if len(frames) < self.max_frames:
            frames += [torch.zeros((3, *self.frame_size), dtype=torch.float32)] * (self.max_frames - len(frames))
            pose_keypoints += [torch.zeros((33, 2), dtype=torch.float32)] * (self.max_frames - len(pose_keypoints))

        # Stack frames and keypoints into tensors
        frames = torch.stack(frames)  # Shape: (max_frames, 3, H, W)
        pose_keypoints = torch.stack(pose_keypoints)  # Shape: (max_frames, 33, 2)

        return frames, pose_keypoints, label



In [2]:
import torch.nn as nn
import torch.nn.functional as F

class VideoClassifier(nn.Module):
    def __init__(self):
        super(VideoClassifier, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten()
        )
        self.fc_keypoints = nn.Sequential(
            nn.Linear(33 * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 84)
        )
        self.classifier = nn.Linear(84 + 32 * 16 * 16, 3)  # Adjust based on input dimensions

    def forward(self, frames, keypoints):
        batch_size, time_steps, _, _, _ = frames.size()

        # Process frames through CNN
        frame_features = [self.cnn(frames[:, t]) for t in range(time_steps)]
        frame_features = torch.stack(frame_features, dim=1).mean(dim=1)  # Aggregate features across time

        # Process keypoints
        keypoints = keypoints.view(batch_size, time_steps, -1)
        keypoint_features = self.fc_keypoints(keypoints.mean(dim=1))

        # Concatenate frame and keypoint features
        combined_features = torch.cat((frame_features, keypoint_features), dim=1)
        
        # Classification
        output = self.classifier(combined_features)
        return F.softmax(output, dim=1)


In [3]:
import torch.optim as optim
from sklearn.metrics import accuracy_score

# Initialize model, loss, and optimizer
model = VideoClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for frames, keypoints, labels in train_loader:
        frames, keypoints, labels = frames.to(device), keypoints.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(frames, keypoints)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Testing loop
def test(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for frames, keypoints, labels in test_loader:
            frames, keypoints, labels = frames.to(device), keypoints.to(device), labels.to(device)
            outputs = model(frames, keypoints)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return accuracy_score(all_labels, all_preds)

# Example of using DataLoader and training/testing loops
data_directory = './dataset'
dataset = PunchingBagDataset(data_directory)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

num_epochs = 15
for epoch in range(num_epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    test_acc = test(model, test_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Test Accuracy: {test_acc:.4f}")


c:\Users\bpohl\anaconda3\envs\aml\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Epoch 1/15, Train Loss: 1.0960, Test Accuracy: 0.4889


c:\Users\bpohl\anaconda3\envs\aml\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Epoch 2/15, Train Loss: 1.0622, Test Accuracy: 0.4444


c:\Users\bpohl\anaconda3\envs\aml\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


...

Epoch 13/15, Train Loss: 0.8501, Test Accuracy: 0.5778


c:\Users\bpohl\anaconda3\envs\aml\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Epoch 14/15, Train Loss: 0.8650, Test Accuracy: 0.6667


c:\Users\bpohl\anaconda3\envs\aml\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Epoch 15/15, Train Loss: 0.8094, Test Accuracy: 0.6000
